In [0]:
from pyspark.sql.functions import current_timestamp,col,sha2, concat_ws
from delta.tables import DeltaTable

In [0]:
df = spark.read.parquet("/Volumes/nyc_taxi/lakehouse/raw/")

In [0]:
bronze_df = df.withColumn("source_file", col("_metadata.file_path")).withColumn("ingestion_timestamp", current_timestamp()).withColumnRenamed("tpep_pickup_datetime", "pickup_datetime").withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")

In [0]:


bronze_df = bronze_df.withColumn(
    "trip_id",
    sha2(
        concat_ws(
            "|",
            col("pickup_datetime"),
            col("dropoff_datetime"),
            col("PULocationID"),
            col("DOLocationID")
        ),
        256
    )
)

In [0]:
bronze_df = bronze_df.dropDuplicates(["trip_id"])

In [0]:
# bronze_df.write\
# .format("delta").mode("append").saveAsTable("nyc_taxi.lakehouse.bronze_taxi")


In [0]:
# %sql
# select count(*) from nyc_taxi.lakehouse.bronze_taxi

In [0]:
bronze_table = DeltaTable.forName(spark,'nyc_taxi.lakehouse.bronze_taxi')
(
    bronze_table.alias("target")
    .merge(bronze_df.alias("source"),"target.trip_id = source.trip_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
